In [ ]:
import logging

from ifc_extractor import ExtractionConfig, CleaningOptions, run_pipeline

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [ ]:
# CONFIGURATION

config = ExtractionConfig(
    input_folder=r"C:\Projects\BuildingA\IFC",
    output_folder=r"C:\Projects\BuildingA\Extracts",

    ifc_queue=[
        "Building-A.ifc"
    ],

    # The anchor is the element every extraction is centered on - stairs by
    # default, but any IFC type works (e.g. "IfcRamp").
    anchor_type="IfcStair",
    target_types=[
        "IfcWall",
        "IfcMember",
        "IfcRailing",
        "IfcSlab",
        "IfcStairFlight",
        "IfcSpace",
        "IfcColumn",
        "IfcFurniture"
    ],

    # Optional: restrict the run to specific elements instead of every
    # anchor_type instance in the file - e.g. to reprocess just a couple
    # of stairs that were skipped last time. IDs don't need to be of
    # anchor_type, and one not found in a given file is just logged and
    # skipped (handy if the list spans multiple files in ifc_queue).
    anchor_guids=[
        # "3yXQETFR9E7fJJ3W_p9jMI",
        # "3g_w9FQb137fh_Ph$XHlzO",
        # "33QnKJ4Nr0GxgpYYgzwUFA",
        # "3yXQETFR9E7fJJ3W_p9jMI",
        # "3eAgA2WBD5ZPF5csCEXihm"
    ],

    proximity_distance=0.50,

    # If a specific element's geometry makes IfcOpenShell hang (some
    # malformed stairs are known to trigger this), it's dropped after this
    # many seconds and the run keeps going: a stuck candidate element (a
    # wall, space, etc. checked for proximity) is excluded from the
    # proximity search, while a stuck anchor itself is skipped entirely.
    # Everything skipped is listed at the end of the run. Skipped anchors
    # are not marked done, so a later run will retry them.
    # Set to None to disable the timeout and wait indefinitely instead.
    anchor_timeout_seconds=120,

    # Covers only an anchor's proximity search + write-to-disk step, once
    # its geometry is already resolved - kept separate from
    # anchor_timeout_seconds because this step never triangulates
    # anything, so a large-but-healthy extraction shouldn't be held to the
    # same tight budget. Raise this (or set it to None) if large
    # extractions are being skipped as "timed out" when they were only
    # ever slow to write, not hung.
    finalize_timeout_seconds=120,

    # Before triangulating a candidate element, first check whether it's
    # anywhere near an anchor using only its (cheap) placement origin -
    # orders of magnitude cheaper than triangulating, so this both speeds
    # up large models and reduces exposure to whatever specific element
    # might hang. It's a conservative filter, not an exact one - a
    # placement is a single point, not the element's true extent - so
    # placement_prefilter_margin exists to absorb that error. Raise the
    # margin if your model has unusually large elements, or set
    # use_placement_prefilter=False to disable it entirely.
    use_placement_prefilter=False,
    placement_prefilter_margin=10.0,

    # Every flag defaults to False: the original unconditional cleanup was
    # never verified not to affect geometry (see tests/test_cleaning.py).
    # Enable what you actually need.
    cleaning=CleaningOptions(
        remove_materials=True,
        remove_styles=True,
        remove_owner_history=True,
    ),
)

In [ ]:
run_pipeline(config)